#Reading from Bronze Table

In [0]:
df=spark.table("workspace.bronze.crm_cust_info")

#Import

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#Data Transformations

In [0]:
#trim all the columns with string value
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


In [0]:
#df.display()

#Normalization

In [0]:
df= (
  df
  .withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status"))=="S","Single")
    .when(F.upper(F.col("cst_marital_status"))=="M","Married")
    .otherwise("n/a")
  )
  .withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr"))=="F","Female")
    .when(F.upper(F.col("cst_gndr"))=="M","Male")
    .otherwise("n/a")
  )
)

In [0]:
df.display()

#Renaming the columns

In [0]:
RENAME_MAP={
  "cst_id":"customer_id",
  "cst_key":"customer_key",
  "cst_firstname":"firstname",
  "cst_lastname":"lastname",
  "cst_marital_status":"marital_status",
  "cst_gndr":"gender",
  "cst_create_date":"_created_date"
}

In [0]:
for oldname,newname in RENAME_MAP.items():
    df=df.withColumnRenamed(oldname,newname)


In [0]:
df.display()

#Write Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_customers")

In [0]:
%sql
select * from workspace.silver.crm_customers